# Step 5: Dataset EDA for BSPC Manuscript

This notebook is the single EDA and manuscript-statistics workflow for the final patient-level CAD/CVD datasets produced by Step 4.

It computes cohort counts, age statistics, sex/comorbidity summaries, integrity checks, waveform-window availability, and paper-ready plots.

In [ ]:
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIG_DIR = REPORTS_DIR / "figures" / "dataset_eda"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CAD_FILE = DATA_DIR / "patient_level_CAD_vs_NonCAD_strict.csv"
CVD_FILE = DATA_DIR / "patient_level_CVD_vs_NonCVD.csv"
PPG_COLS = [f"ppg_min_{i}" for i in range(3, 9)]
FLAG_COLS = ["is_diabetic", "has_high_cholesterol", "is_obese"]

PROJECT_ROOT

## Helper Functions

In [ ]:
def metadata_columns(csv_path):
    cols = pd.read_csv(csv_path, nrows=0).columns.tolist()
    return [col for col in cols if col not in PPG_COLS]

def load_metadata(csv_path):
    return pd.read_csv(csv_path, usecols=metadata_columns(csv_path))

def corrected_age(series):
    return series.where(series <= 89, 90)

def ci95(series):
    clean = series.dropna()
    if len(clean) <= 1:
        return np.nan, np.nan
    return stats.t.interval(0.95, len(clean) - 1, loc=clean.mean(), scale=stats.sem(clean))

def cohens_d(pos, neg):
    n1, n2 = len(pos), len(neg)
    s1, s2 = pos.std(ddof=1), neg.std(ddof=1)
    pooled = math.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    return (pos.mean() - neg.mean()) / pooled

def welch_df(pos, neg):
    n1, n2 = len(pos), len(neg)
    s1, s2 = pos.std(ddof=1), neg.std(ddof=1)
    v1, v2 = s1**2 / n1, s2**2 / n2
    return (v1 + v2) ** 2 / ((v1**2 / (n1 - 1)) + (v2**2 / (n2 - 1)))

def binary_count(series):
    return int(pd.to_numeric(series, errors="coerce").fillna(0).sum())

def male_count(df):
    if "GENDER" in df.columns:
        return int((df["GENDER"].astype(str).str.upper() == "M").sum())
    if "GENDER_NUM" in df.columns:
        return int((pd.to_numeric(df["GENDER_NUM"], errors="coerce") == 1).sum())
    return 0

def group_summary(df, label_col, label_value, group_name):
    group = df[df[label_col] == label_value].copy()
    age = group["AGE_CORR"].dropna()
    lo, hi = ci95(age)
    out = {
        "group": group_name,
        "n": int(len(group)),
        "age_mean": round(float(age.mean()), 2),
        "age_sd": round(float(age.std(ddof=1)), 2),
        "age_ci_low": round(float(lo), 2),
        "age_ci_high": round(float(hi), 2),
        "male_n": male_count(group),
        "male_pct": round(100 * male_count(group) / len(group), 2),
    }
    for col in FLAG_COLS:
        count = binary_count(group[col])
        out[f"{col}_n"] = count
        out[f"{col}_pct"] = round(100 * count / len(group), 2)
    return out

def cohort_stats(df, label_col, pos_name, neg_name):
    pos = df[df[label_col] == 1]["AGE_CORR"].dropna()
    neg = df[df[label_col] == 0]["AGE_CORR"].dropna()
    test = stats.ttest_ind(pos, neg, equal_var=False)
    return {
        "total_n": int(len(df)),
        "unique_subjects": int(df["SUBJECT_ID"].nunique()),
        "duplicate_subjects": int(df.duplicated("SUBJECT_ID").sum()),
        "class_counts": {str(k): int(v) for k, v in df[label_col].value_counts().sort_index().items()},
        "positive": group_summary(df, label_col, 1, pos_name),
        "negative": group_summary(df, label_col, 0, neg_name),
        "age_comparison": {
            "welch_t": round(float(test.statistic), 4),
            "welch_df": round(float(welch_df(pos, neg)), 2),
            "p_value": float(test.pvalue),
            "cohens_d": round(float(cohens_d(pos, neg)), 4),
        },
    }

def waveform_window_stats(csv_path):
    ppg_cols = [c for c in pd.read_csv(csv_path, nrows=0).columns if c.startswith("ppg_min_")]
    df = pd.read_csv(csv_path, usecols=ppg_cols)
    non_missing = df.notna() & df.ne("")
    return {
        "source_file": str(csv_path),
        "records": int(len(df)),
        "ppg_columns": ppg_cols,
        "available_waveform_windows": int(non_missing.sum().sum()),
        "records_with_all_ppg_windows": int(non_missing.all(axis=1).sum()),
        "mean_available_windows_per_record": round(float(non_missing.sum(axis=1).mean()), 2),
    }

## Load Final Patient-Level Datasets

In [ ]:
cad = load_metadata(CAD_FILE)
cvd = load_metadata(CVD_FILE)

cad["AGE_CORR"] = corrected_age(cad["AGE"])
cvd["AGE_CORR"] = corrected_age(cvd["AGE"])

print("CAD:", cad.shape, "unique subjects:", cad["SUBJECT_ID"].nunique())
print("CVD:", cvd.shape, "unique subjects:", cvd["SUBJECT_ID"].nunique())
display(cad.head())
display(cvd.head())

## Compute Manuscript Statistics

In [ ]:
results = {
    "source_files": {
        "cad": str(CAD_FILE),
        "cvd": str(CVD_FILE),
    },
    "CAD": cohort_stats(cad, "CAD_LABEL", "CAD+", "Non-CAD"),
    "CVD": cohort_stats(cvd, "CVD_LABEL", "CVD+", "Non-CVD"),
    "integrity": {
        "cad_subjects_subset_of_cvd_subjects": bool(set(cad["SUBJECT_ID"]).issubset(set(cvd["SUBJECT_ID"]))),
        "cad_controls_equal_cvd_controls": bool(set(cad.loc[cad["CAD_LABEL"] == 0, "SUBJECT_ID"]) == set(cvd.loc[cvd["CVD_LABEL"] == 0, "SUBJECT_ID"])),
        "cad_fs_values": sorted(cad["fs"].dropna().unique().tolist()) if "fs" in cad else [],
        "cvd_fs_values": sorted(cvd["fs"].dropna().unique().tolist()) if "fs" in cvd else [],
    },
    "waveform_windows": waveform_window_stats(CVD_FILE),
}

rows = []
for cohort_key in ["CAD", "CVD"]:
    for group_key in ["positive", "negative"]:
        rows.append({"cohort": cohort_key, **results[cohort_key][group_key]})
summary_df = pd.DataFrame(rows)
display(summary_df)

## Save JSON and CSV Reports

In [ ]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

with open(REPORTS_DIR / "dataset_eda_summary.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

summary_df.to_csv(REPORTS_DIR / "dataset_eda_summary.csv", index=False)

with open(REPORTS_DIR / "final_manuscript_stats.json", "w", encoding="utf-8") as f:
    json.dump({"CAD": results["CAD"], "CVD": results["CVD"], "waveform_windows": results["waveform_windows"]}, f, indent=2)

print(REPORTS_DIR / "dataset_eda_summary.json")
print(REPORTS_DIR / "dataset_eda_summary.csv")
print(REPORTS_DIR / "final_manuscript_stats.json")

## Plot Class Balance

In [ ]:
class_balance = pd.concat([
    cad["CAD_LABEL"].map({1: "CAD+", 0: "Non-CAD"}).value_counts().rename_axis("group").reset_index(name="patients").assign(cohort="Strict CAD"),
    cvd["CVD_LABEL"].map({1: "CVD+", 0: "Non-CVD"}).value_counts().rename_axis("group").reset_index(name="patients").assign(cohort="Broad CVD"),
], ignore_index=True)

fig, ax = plt.subplots(figsize=(6.5, 4.2))
sns.barplot(data=class_balance, x="cohort", y="patients", hue="group", ax=ax)
ax.set_xlabel("")
ax.set_ylabel("Patients")
ax.set_title("Patient-Level Class Balance")
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3, fontsize=8)
ax.legend(title="Group", frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "class_balance.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "class_balance.pdf", bbox_inches="tight")
plt.show()

## Plot Age Distributions

In [ ]:
age_plot = pd.concat([
    cad[cad["CAD_LABEL"] == 1].assign(group="CAD+")[["AGE_CORR", "group"]],
    cad[cad["CAD_LABEL"] == 0].assign(group="Non-CAD / Non-CVD")[["AGE_CORR", "group"]],
    cvd[cvd["CVD_LABEL"] == 1].assign(group="CVD+")[["AGE_CORR", "group"]],
], ignore_index=True)

fig, ax = plt.subplots(figsize=(7.2, 4.4))
sns.histplot(data=age_plot, x="AGE_CORR", hue="group", bins=np.arange(20, 96, 5), element="step", common_norm=False, ax=ax)
ax.set_xlabel("Age at ICU Admission (years; >89 coded as 90)")
ax.set_ylabel("Patients")
ax.set_title("Age Distribution by Disease Group")
fig.tight_layout()
fig.savefig(FIG_DIR / "age_distribution.png", bbox_inches="tight")
fig.savefig(FIG_DIR / "age_distribution.pdf", bbox_inches="tight")
plt.show()

## Plot Sex and Comorbidity Rates

In [ ]:
rates = []
for cohort_name, df, label_col, mapping in [
    ("Strict CAD", cad, "CAD_LABEL", {1: "CAD+", 0: "Non-CAD"}),
    ("Broad CVD", cvd, "CVD_LABEL", {1: "CVD+", 0: "Non-CVD"}),
]:
    for label_value, group_name in mapping.items():
        group = df[df[label_col] == label_value]
        rates.append({"cohort": cohort_name, "group": group_name, "variable": "Male", "percent": 100 * male_count(group) / len(group)})
        for col, name in [("is_diabetic", "Diabetes"), ("has_high_cholesterol", "High cholesterol"), ("is_obese", "Obesity")]:
            rates.append({"cohort": cohort_name, "group": group_name, "variable": name, "percent": 100 * binary_count(group[col]) / len(group)})

rate_df = pd.DataFrame(rates)
grid = sns.catplot(data=rate_df, x="variable", y="percent", hue="group", col="cohort", kind="bar", height=4, aspect=1.08, sharey=True)
grid.set_axis_labels("", "Patients (%)")
grid.set_titles("{col_name}")
for ax in grid.axes.flat:
    ax.tick_params(axis="x", rotation=25)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f", padding=2, fontsize=7)
grid.fig.tight_layout()
grid.fig.savefig(FIG_DIR / "sex_comorbidity_rates.png", bbox_inches="tight")
grid.fig.savefig(FIG_DIR / "sex_comorbidity_rates.pdf", bbox_inches="tight")
plt.show()

## Plot Waveform Quality Examples

This figure exports representative accepted-versus-rejected PPG waveform examples as manuscript-ready PNG/PDF files.

In [ ]:

# Representative accepted/rejected waveform examples for the manuscript.
# These SUBJECT_IDs are expected in the final CVD patient-level file used for the manuscript example.
EXAMPLE_SUBJECTS = {
    "Accepted waveform": 97476,
    "Rejected waveform": 54934,
}
EXAMPLE_COL = "ppg_min_8"

def parse_ppg_window(value):
    if pd.isna(value):
        return np.array([], dtype=float)
    return np.fromstring(str(value).replace("[", " ").replace("]", " ").replace(",", " "), sep=" ")

def load_example_windows(csv_path, subject_map, ppg_col):
    needed = set(subject_map.values())
    usecols = ["SUBJECT_ID", "fs", ppg_col]
    rows = []
    for chunk in pd.read_csv(csv_path, usecols=usecols, chunksize=512):
        hit = chunk[chunk["SUBJECT_ID"].isin(needed)]
        if not hit.empty:
            rows.append(hit)
    if not rows:
        raise ValueError(f"None of the requested subjects were found: {sorted(needed)}")
    found = pd.concat(rows, ignore_index=True).drop_duplicates("SUBJECT_ID")
    missing = needed - set(found["SUBJECT_ID"])
    if missing:
        raise ValueError(f"Missing requested subjects in {csv_path}: {sorted(missing)}")
    return found.set_index("SUBJECT_ID")

example_rows = load_example_windows(CVD_FILE, EXAMPLE_SUBJECTS, EXAMPLE_COL)

fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.1), sharex=False)
plot_specs = [
    ("Accepted waveform", "#0e3b75"),
    ("Rejected waveform", "#b22222"),
]

for ax, (title, color) in zip(axes, plot_specs):
    subject_id = EXAMPLE_SUBJECTS[title]
    fs = int(example_rows.loc[subject_id, "fs"])
    signal = parse_ppg_window(example_rows.loc[subject_id, EXAMPLE_COL])
    if signal.size == 0:
        raise ValueError(f"No waveform samples found for SUBJECT_ID {subject_id}, {EXAMPLE_COL}")
    t = np.arange(signal.size) / fs
    step = max(1, int(round(fs * 0.2)))
    ax.plot(t[::step], signal[::step], color=color, linewidth=1.25)
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.set_xlabel("Time (s)", fontsize=8)
    ax.set_ylabel("Amplitude", fontsize=8)
    ax.grid(True, color="#D0D0D0", linewidth=0.45, alpha=0.8)
    ax.tick_params(axis="both", labelsize=7)
    ax.set_xlim(0, 60)
    ax.text(
        0.02,
        0.96,
        f"SUBJECT_ID {subject_id}\n{EXAMPLE_COL}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=7,
        bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="#BBBBBB", alpha=0.85),
    )

fig.tight_layout(rect=[0, 0, 1, 0.92])

waveform_png = FIG_DIR / "waveform_quality_examples.png"
waveform_pdf = FIG_DIR / "waveform_quality_examples.pdf"
fig.savefig(waveform_png, bbox_inches="tight", dpi=600)
fig.savefig(waveform_pdf, bbox_inches="tight")

plt.show()
print(waveform_png)
print(waveform_pdf)


## Manuscript-Ready Summary

In [ ]:
for key, label in [("CAD", "Strict CAD"), ("CVD", "Broad CVD")]:
    data = results[key]
    pos = data["positive"]
    neg = data["negative"]
    comp = data["age_comparison"]
    print(f"{label}: {data['total_n']:,} unique patients")
    print(f"  {pos['group']}: n={pos['n']:,}, age={pos['age_mean']:.1f} +/- {pos['age_sd']:.1f} years [{pos['age_ci_low']:.1f}-{pos['age_ci_high']:.1f}], male={pos['male_pct']:.1f}%")
    print(f"  {neg['group']}: n={neg['n']:,}, age={neg['age_mean']:.1f} +/- {neg['age_sd']:.1f} years [{neg['age_ci_low']:.1f}-{neg['age_ci_high']:.1f}], male={neg['male_pct']:.1f}%")
    print(f"  Welch t({comp['welch_df']:.0f})={comp['welch_t']:.1f}, p={comp['p_value']:.2e}, Cohen's d={comp['cohens_d']:.2f}")
    print(f"  Diabetes: {pos['is_diabetic_pct']:.1f}% vs {neg['is_diabetic_pct']:.1f}%")
    print(f"  High cholesterol: {pos['has_high_cholesterol_pct']:.1f}% vs {neg['has_high_cholesterol_pct']:.1f}%")
    print(f"  Obesity: {pos['is_obese_pct']:.1f}% vs {neg['is_obese_pct']:.1f}%")
    print()